In [ ]:
!pip show bitsandbytes
!pip install -U bitsandbytes
!pip show bitsandbytes
!pip show transformers

# 1. 단어 사전 기반 매핑 복원

## 1) Import

In [ ]:
import pandas as pd

## 2) Data Load

In [ ]:
train = pd.read_csv('./train.csv', encoding = 'utf-8-sig')
test = pd.read_csv('./test.csv', encoding = 'utf-8-sig')

## 3) 단어 사전 생성

In [ ]:
match_dict = {}

for input_text, output_text in zip(train['input'], train['output']):
    input_words = input_text.split()
    output_words = output_text.split()
    for iw, ow in zip(input_words, output_words):
        match_dict[iw] = ow

## 4) 변환 적용

In [ ]:
def replace_words(input_text, match_dict):
    words = input_text.split()
    replaced_words = [match_dict.get(word, word) for word in words]
    return " ".join(replaced_words)

converted_reivews = test['input'].apply(lambda x: replace_words(x, match_dict)).tolist()

## 5) Submission

In [ ]:
submission = pd.read_csv('./sample_submission.csv', encoding = 'utf-8-sig')
submission['output'] = converted_reivews
submission.to_csv('./submission.csv', index = False, encoding = 'utf-8')

# 2. LLM활용 (Gemma)

## 1) Import

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
from tqdm import tqdm

## 2) Data Load

In [ ]:
train = pd.read_csv('/kaggle/input/dacon-encrypted-dataset/train.csv', encoding = 'utf-8')
test = pd.read_csv('/kaggle/input/dacon-encrypted-dataset/test.csv', encoding = 'utf-8')

In [ ]:
train = train[:2000]
test = test[:100]

In [ ]:
samples = []

for i in range(10):
    sample = f"input : {train['input'][i]} \n output : {train['output'][i]}"
    samples.append(sample)

In [ ]:
samples

## 3) Model load

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit = True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_use_double_quant = True,
    bnb_4bit_compute_dtype=torch.bfloat16
)
model_id = 'beomi/gemma-ko-2b'

tokenizer = AutoTokenizer.from_pretrained(model_id)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = 'right'

device_map = {"": [0, 1]} # 지금 gpu가 T4 ×2이므로 두 gpu모두 사용기기

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    quantization_config=bnb_config,
    torch_dtype = torch.bfloat16,
    device_map='auto'
)

## 4) Inference

In [ ]:
import torch
from tqdm.notebook import tqdm

batch_size = 5
restored_reviews = []

# --------------------------------------------------------
# 🔥 개선된 길이 보정 함수
# --------------------------------------------------------
def fix_length_improved(input_text, output_text):
    """입력과 출력의 길이를 맞추되, 더 자연스럽게 처리"""
    input_len = len(input_text)
    output_len = len(output_text)
    
    if output_len == input_len:
        return output_text
    
    # output이 너무 길면 자름
    if output_len > input_len:
        # 문장 부호를 고려하여 자르기
        truncated = output_text[:input_len]
        return truncated
    
    # output이 짧으면 공백으로 채움 (마지막 글자 반복보다 자연스러움)
    if output_len < input_len:
        diff = input_len - output_len
        return output_text + ' ' * diff


# --------------------------------------------------------
# 🔥 간결하고 효과적인 Few-shot 예시
# --------------------------------------------------------
fewshot_examples = """다음은 오염된 한국어를 복원하는 예시입니다:

곤욱삐갸 삐쌀침만 좋얏뎐 쿄씸? → 교육비가 비싸지만 좋았던 곳임?
쩌은멘 춈 흼믹했눈떠 났쭝웬 쾌 익쑥행쥠. → 처음엔 좀 희미했는데 나중엔 꽤 익숙해짐.
앍뚤한 삐듭팩끼 끝냔 겉 갓뎐뎨? → 알뜰한 피드백이 끝난 것 같던데?

오염된 텍스트를 원래 의미의 자연스러운 한국어로 복원하세요. 글자 수는 원본과 동일하게 유지하세요.
"""


# --------------------------------------------------------
# 🔥 개선된 후처리 함수
# --------------------------------------------------------
def clean_output(output_text, input_text):
    """생성된 출력을 깔끔하게 정리"""
    restored = output_text.strip()
    
    # 1. 화살표(→) 이후 텍스트만 추출
    if '→' in restored:
        restored = restored.split('→', 1)[1].strip()
    
    # 2. 따옴표 제거
    restored = restored.strip('"\'')
    
    # 3. 불필요한 키워드가 포함된 경우 그 이전까지만 사용
    for keyword in ['Input:', 'Output:', '오염된', '복원', '예시', '\n']:
        if keyword in restored:
            restored = restored.split(keyword)[0].strip()
    
    # 4. 빈 문자열인 경우 입력 반환
    if not restored:
        restored = input_text
    
    return restored


# --------------------------------------------------------
# 🔥 Batch Inference Loop (개선 버전)
# --------------------------------------------------------
for start in tqdm(range(0, len(test), batch_size)):
    end = min(start + batch_size, len(test))
    batch = test.iloc[start:end]

    # 간결한 프롬프트 구성
    prompts = [
        f"{fewshot_examples}\n{row['input']} →"
        for _, row in batch.iterrows()
    ]

    # Tokenization
    inputs = tokenizer(
        prompts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=512  # 최대 길이 명시
    ).to("cuda")

    # max_new_tokens: 여유 공간 확보
    max_input_len = max(len(row["input"]) for _, row in batch.iterrows())
    max_new_tokens = max_input_len + 10

    # 모델 배치 생성 (개선된 파라미터)
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,              # sampling 활성화
            temperature=0.7,             # 적절한 다양성
            top_p=0.9,                   # nucleus sampling
            repetition_penalty=1.5,      # 강화된 반복 억제
            eos_token_id=tokenizer.eos_token_id,
            pad_token_id=tokenizer.pad_token_id
        )

    # 입력 토큰 길이
    input_len = inputs.input_ids.shape[1]

    # 새로 생성된 토큰만 추출 및 디코딩
    generated_ids = output_ids[:, input_len:]
    outputs = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)

    # 후처리: 출력 정리 + 길이 보정
    for idx, out in enumerate(outputs):
        input_text = batch.iloc[idx]["input"]
        
        # 1. 출력 정리
        cleaned = clean_output(out, input_text)
        
        # 2. 길이 보정
        fixed = fix_length_improved(input_text, cleaned)
        
        restored_reviews.append(fixed)

### 5) Submission

In [ ]:
submission = pd.read_csv('/kaggle/input/dacon-encrypted-dataset/sample_submission.csv', encoding = 'utf-8')
submission['output'] = restored_reviews
submission.to_csv('/kaggle/working/submission.csv', index = False, encoding = 'utf-8-sig')